# AI Video Creator

Generate a narrated, scored, cinematic video from a single text theme.

**Pipeline:** theme &rarr; story (Mistral-7B) &rarr; scene images (Stable Diffusion) &rarr; narration (Bark) &rarr; background music (Stable Audio Open) &rarr; assembled 1080p MP4 (MoviePy).

**Before you run:** Runtime &rarr; Change runtime type &rarr; select a GPU (T4 or better). This pipeline downloads several multi-GB models and will not run in reasonable time on CPU.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


## 2. Mount Google Drive

Used to persist generated videos (and optionally cache model weights) across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AI_Video_Creator_Output'
import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print('Drive output folder:', DRIVE_OUTPUT_DIR)


## 3. Get the project code

Pick ONE of the two options below.

- **Option A**: clone from your GitHub repo (fill in `REPO_URL`).
- **Option B**: you already uploaded/synced the `ai-video-creator` folder into this Colab's `/content` (e.g. via Drive) — just set `PROJECT_DIR` to that path.

In [ ]:
# --- Option A: clone from GitHub ---
REPO_URL = ''  # e.g. 'https://github.com/<your-username>/ai-video-creator.git'
PROJECT_DIR = '/content/ai-video-creator'

import os
if REPO_URL:
    if not os.path.isdir(PROJECT_DIR):
        !git clone "$REPO_URL" "$PROJECT_DIR"
    else:
        print('Project dir already exists, pulling latest changes...')
        !cd "$PROJECT_DIR" && git pull
else:
    # --- Option B: folder already present locally (e.g. copied from Drive) ---
    assert os.path.isdir(PROJECT_DIR), (
        f'{PROJECT_DIR} does not exist and REPO_URL is empty. '
        'Either set REPO_URL above, or copy the ai-video-creator folder to this path '
        '(e.g. !cp -r /content/drive/MyDrive/ai-video-creator /content/).'
    )

%cd $PROJECT_DIR
print('Using project directory:', PROJECT_DIR)


## 4. Install dependencies

This installs everything in `requirements.txt` plus ImageMagick (needed by MoviePy for subtitle rendering).

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq update && apt-get -qq install -y imagemagick > /dev/null

# Relax ImageMagick's default policy so MoviePy can render text (subtitles).
# Safe in an ephemeral Colab VM; do not do this on a shared/production machine.
!sed -i 's/rights="none" pattern="@\*"/rights="read|write" pattern="@*"/' /etc/ImageMagick-6/policy.xml || true
print('Dependencies installed.')


## 5. Configure Hugging Face access

Some models (Stable Diffusion, Stable Audio Open) are gated and require a Hugging Face token, plus accepting the license on the model page at least once.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass('Enter your Hugging Face token (input hidden): ')

os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face Hub.')


## 6. Run the pipeline

Set your theme, output filename, style, and quality preset below, then run.
Each stage (story / images / narration / music / video) prints progress and
logs to `output/logs/pipeline.log`. GPU memory is cleared between stages.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from main import create_video

THEME = "A cyberpunk city with neon lights and flying cars at night"
OUTPUT_NAME = "cyberpunk_city.mp4"
STYLE = "cinematic"       # cinematic | concept_art | photorealistic | anime | noir
QUALITY = "balanced"      # draft | balanced | high

try:
    video_path = create_video(
        theme=THEME,
        output_name=OUTPUT_NAME,
        style=STYLE,
        quality=QUALITY,
    )
    print('\nSuccess! Video saved to:', video_path)
except Exception as e:
    print('\nPipeline failed:', e)
    print('Check output/logs/pipeline.log for full details.')
    print('Common fixes: lower QUALITY to "draft", ensure HF_TOKEN has accepted model licenses, '
          'or re-run this cell (transient download/OOM errors are retried automatically per-stage).')
    raise


## 7. Preview the result

In [ ]:
from IPython.display import Video
Video(str(video_path), embed=True, width=640)


## 8. Save output to Google Drive

In [ ]:
import shutil

dest = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(str(video_path)))
shutil.copy(str(video_path), dest)
print('Copied final video to Drive:', dest)

# Optionally also persist the intermediate assets (story JSON, images, audio):
assets_dest = os.path.join(DRIVE_OUTPUT_DIR, 'assets_' + os.path.splitext(os.path.basename(str(video_path)))[0])
shutil.copytree(os.path.join(PROJECT_DIR, 'output'), assets_dest, dirs_exist_ok=True)
print('Copied intermediate assets to Drive:', assets_dest)


## 9. (Optional) Batch processing

Run several themes back-to-back in one session. Each call fully unloads its models between themes.

In [ ]:
THEMES = [
    # ("A serene Japanese garden through the four seasons", "japanese_garden.mp4"),
    # ("A deep-sea expedition discovering bioluminescent creatures", "deep_sea.mp4"),
]

batch_results = []
for theme, output_name in THEMES:
    try:
        path = create_video(theme=theme, output_name=output_name, style=STYLE, quality=QUALITY)
        shutil.copy(str(path), os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(str(path))))
        batch_results.append((theme, path, 'ok'))
    except Exception as e:
        batch_results.append((theme, None, f'failed: {e}'))

for theme, path, status in batch_results:
    print(f'{status:40s} | {theme}')
